In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Data Preparation

In [16]:
# Load all datasets
customers = pd.read_csv("../source_data/olist_customers_dataset.csv")
orders = pd.read_csv("../source_data/olist_orders_dataset.csv")
order_items = pd.read_csv("../source_data/olist_order_items_dataset.csv")
products = pd.read_csv("../source_data/olist_products_dataset.csv")
sellers = pd.read_csv("../source_data/olist_sellers_dataset.csv")
payments = pd.read_csv("../source_data/olist_order_payments_dataset.csv")
geo = pd.read_csv("../source_data/olist_geolocation_dataset.csv")
category = pd.read_csv("../source_data/product_category_name_translation.csv")

In [17]:
# Merge orders with customers to get customer details in the orders dataframe
df = orders.copy()
df = df.merge(customers, on="customer_id", how="left")

In [18]:
# Merge orders with order_items to get product details in the orders dataframe
df = df.merge(order_items, on="order_id", how="left")
# Merge orders with products to get product details
df = df.merge(products, on="product_id", how="left")
# Merge orders with category to get category details
df = df.merge(category, on="product_category_name", how="left")
# Merge orders with sellers to get seller details
df = df.merge(sellers, on="seller_id", how="left")
# Merge orders with payments to get payment details
df = df.merge(payments, on="order_id", how="left")

In [19]:
# Aggregate geolocation data to get average latitude and longitude for each zip code prefix
geography = geo.groupby("geolocation_zip_code_prefix").agg({
    "geolocation_lat": "mean",
    "geolocation_lng": "mean"
}).reset_index()

In [20]:
# Merge geolocation data with the main dataframe to get customer latitude and longitude based on their zip code prefix
df = df.merge(
    geography,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
).rename(columns={
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng"
}).drop(columns=["geolocation_zip_code_prefix"])

In [21]:
# Merge geolocation data with the main dataframe to get seller latitude and longitude based on their zip code prefix
df = df.merge(
    geography,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
).rename(columns={
    "geolocation_lat": "seller_lat",
    "geolocation_lng": "seller_lng"
}).drop(columns=["geolocation_zip_code_prefix"])

In [22]:
df_final = df.groupby("order_id").agg({
    "price": "sum",
    "freight_value": "sum",
    "payment_value": "sum",
    "payment_installments": "max",
    "product_weight_g": "mean",
    
    "customer_state": "first",
    "seller_state": "first",
    
    "customer_lat": "first",
    "customer_lng": "first",
    "seller_lat": "first",
    "seller_lng": "first",
    
    "order_purchase_timestamp": "first",
    "order_approved_at": "first",
    "order_delivered_carrier_date": "first",
    "order_estimated_delivery_date": "first",
    "order_delivered_customer_date": "first"
}).reset_index()

Why use "first"? Because these details apply to the whole order and generally don't change from item to item. An order only has one purchase time and one delivery destination. Telling Pandas to Keep this data as it is; don't try to add it up.

In [23]:
# Calculate delivery delay in days
df_final["delayed"] = (
    pd.to_datetime(df_final["order_delivered_customer_date"]) >
    pd.to_datetime(df_final["order_estimated_delivery_date"])
).astype(int)

In [24]:
df_final.head()

,order_id,price,freight_value,payment_value,payment_installments,product_weight_g,customer_state,seller_state,customer_lat,customer_lng,seller_lat,seller_lng,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_estimated_delivery_date,order_delivered_customer_date,delayed
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19,2.0,650.0,RJ,SP,-21.762775,-41.309633,-22.496953,-44.127492,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-29 00:00:00,2017-09-20 23:43:48,0
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83,3.0,30000.0,SP,SP,-20.220527,-50.903424,-23.565096,-46.518565,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-15 00:00:00,2017-05-12 16:04:24,0
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87,5.0,3050.0,MG,MG,-19.870305,-44.593326,-22.262584,-46.171124,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-02-05 00:00:00,2018-01-22 13:19:16,0
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78,2.0,200.0,SP,SP,-23.089925,-46.611654,-20.553624,-47.387359,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-20 00:00:00,2018-08-14 13:32:39,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04,3.0,3750.0,SP,PR,-23.243402,-46.827614,-22.929384,-53.135873,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-17 00:00:00,2017-03-01 16:42:31,0


In [25]:
# Drop the original delivery date column as we now have the delayed flag
df_final = df_final.drop(columns=["order_delivered_customer_date"])

In [26]:
data = df_final.to_csv("../data/data.csv", index=False)